In [ ]:
using Symbolics

# Building Symbolic Expressions

In [ ]:
@variables x y

In [ ]:
z=x^2+y

In [ ]:
A=[x^2+y 0 2x
   0 0 2y
   y^2 0 0]

In [ ]:
function f(u)
  [u[1] - u[3], u[1]^2 - u[2], u[3] + u[2]]
end
f([x, y, z]) # Recall that z = x^2 + y

In [ ]:
u = Symbolics.variables(:u, 1:5)
f(u)

# Derivatives

In [ ]:
@variables t
D = Differential(t)

In [ ]:
z = t + t^2
D(z)

In [ ]:
expand_derivatives(D(z))

In [ ]:
D.x

In [ ]:
Symbolics.jacobian([x + x*y, x^2 + y], [x, y])

# Simplification and Substitution

In [ ]:
simplify(2x + 2y)

In [ ]:
B = simplify.([t + t^2 + t + t^2  2t + 4t
               x + y + y + 2t     x^2 - x^2 + y^2])

In [ ]:
(substitute.(B, (Dict(x => y^2),)))

In [ ]:
V = substitute.(B, (Dict(x => 2.0, y => 3.0, t => 4.0),))

In [ ]:
Symbolics.value.(V)

# Syms and callable Syms

In [ ]:
@variables t x(t) y(t)

In [ ]:
z = x + y*t
expand_derivatives(D(z))

In [ ]:
@variables g(..)

In [ ]:
z = g(x) + g(y)

In [ ]:
expand_derivatives(D(z))

In [ ]:
x=cos(t)

In [ ]:
expand_derivatives(D(x))

In [ ]:
g=cos

In [ ]:
g(x)

In [ ]:
expand_derivatives(D(g(x)))

# Building Non-Allocating Parallel Functions for Sparse Matrices

In [ ]:
using LinearAlgebra
using SparseArrays

In [ ]:
N = 8
@variables x y
A = sparse(Tridiagonal([x^i for i in 1:N-1], [x^i * y^(8-i) for i in 1:N], [y^i for i in 1:N-1]))
show(A)

In [ ]:
Base.remove_linenums!(build_function(A,[x,y],parallel=Symbolics.MultithreadedForm())[2])

In [ ]:
N = 10
_S = sprand(N, N, 0.1)
_Q = sprand(N, N, 0.1)

F(z) = [ # some complicated sparse amenable function
    _S * z
    _Q * z.^2
]

Symbolics.@variables z[1:N]

sj = Symbolics.sparsejacobian(F(z), z) # sparse jacobian

f_expr = build_function(sj, z)

rows, cols, _ = findnz(sj)
out = sparse(rows, cols, zeros(length(cols)), size(sj)...) # pre-allocate, and correct structure
myf = eval(last(f_expr))
myf(out, rand(N)) # note that out matches the sparsity structure of sj
out

# Mixed Symbolic-Numeric Perturbation Theory

## Solving the quintic equation

In [ ]:
@variables x
quintic = x^5 + x ~ 1

In [ ]:
function solve_newton(eq, x, x₀; abstol=1e-8, maxiters=50)
    # symbolic expressions for f(x) and f′(x)
    f = eq.lhs - eq.rhs # want to find root of f(x)
    f′ = Symbolics.derivative(f, x)

    xₙ = x₀ # numerical value of the initial guess
    for i = 1:maxiters
        # calculate new guess by numerically evaluating symbolic expression at previous guess
        xₙ₊₁ = substitute(x - f / f′, x => xₙ)
        if abs(xₙ₊₁ - xₙ) < abstol
            return xₙ₊₁ # converged
        else
            xₙ = xₙ₊₁
        end
    end
    error("Newton's method failed to converge")
end

x_newton = solve_newton(quintic, x, 1.0)
println("Newton's method solution: x = ", x_newton)

In [ ]:
@variables ϵ # expansion variable
quintic = x^5 + ϵ*x ~ 1

In [ ]:
n=15
x_coeffs, = @variables a[0:n] # create Taylor series coefficients
x_taylor = series(x_coeffs, ϵ) # expand x in a power series in ϵ

In [ ]:
quintic_taylor = substitute(quintic, x => x_taylor)
quintic_taylor = taylor(quintic_taylor, ϵ, 0:n)

In [ ]:
quintic_eqs = taylor_coeff(quintic_taylor, ϵ, 0:n)
quintic_eqs[1:5] # for readability, show only 5 shortest equations

In [ ]:
function solve_cascade(eqs, xs, x₀)
    sol = Dict(xs[begin] => x₀) # store solutions in a map

    # verify that x₀ is a solution of the first equation
    eq0 = substitute(eqs[1], sol)
    isequal(eq0.lhs, eq0.rhs) || error("$sol does not solve $(eqs[1])")

    # solve remaining equations sequentially
    for i in 2:length(eqs)
        eq = substitute(eqs[i], sol) # insert previous solutions
        x = xs[begin+i-1] # need not be 1-indexed
        xsol = Symbolics.symbolic_linear_solve(eq, x) # solve current equation
        sol = merge(sol, Dict(x => xsol)) # store solution
    end

    return sol
end

In [ ]:
x_coeffs_sol = solve_cascade(quintic_eqs, x_coeffs, 1)
x_pert = substitute(x_taylor, x_coeffs_sol)

In [ ]:
for n in 0:n
    x_pert_sol = substitute(taylor(x_pert, ϵ, 0:n), ϵ => 1)
    println("$n-th order solution: x = $x_pert_sol = $(x_pert_sol * 1.0)")
end

## Solving Kepler's equation

In [ ]:
@variables e E M
kepler = E - e * sin(E) ~ M

In [ ]:
vals_earth = Dict(e => 0.01671, M => π/2)
E_newton = solve_newton(substitute(kepler, vals_earth), E, π/2)
println("Newton's method solution: E = ", E_newton)

In [ ]:
E_taylor = series(E, M, 0:5) # this auto-creates coefficients E[0:5]
E_coeffs = taylor_coeff(E_taylor, M) # get a handle to the coefficients
kepler_eqs = taylor_coeff(substitute(kepler, E => E_taylor), M, 0:5)
kepler_eqs[1:4] # for readability

In [ ]:
E_coeffs_sol = solve_cascade(kepler_eqs, E_coeffs, 0, M)
for i in 0:5
    E_coeffs_sol[E[i]] = simplify(E_coeffs_sol[E[i]])
end
E_pert = substitute(E_taylor, E_coeffs_sol)

In [ ]:
for n in 0:5
    println("$n-th order solution: E = ", substitute(taylor(E_pert, M, 0:n), vals_earth))
end